# Class 21: Clusters on Demand
*Elements of Data Science (Honors)*

<br>**<center>Learning Goals**

|Area|Concept|
|---|---|
|Clustering|Sort points into groups using only distance -- no labels|
|k-means|Assign each point to the nearest center, move centers, repeat|
|Skepticism|k-means always returns k groups, real or not|

Write your answers on the handout, not in this notebook.

In [ ]:
import numpy as np
from datascience import *
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

## The k-means tool

Run this cell once. You do **not** need to edit it, but read the comments: Step A and Step B are the two steps on your handout.

In [ ]:
def standard_units(arr):
    return (arr - np.mean(arr)) / np.std(arr)

def kmeans(tbl, k, seed):
    """Sort the rows of a two-column table into k clusters.
    Returns a copy of the table with a new 'cluster' column."""
    # Put both columns in standard units so neither one dominates the distance
    x = standard_units(tbl.column(0))
    y = standard_units(tbl.column(1))

    # Start: pick k rows at random to be the first centers
    np.random.seed(seed)
    start_rows = np.random.choice(np.arange(tbl.num_rows), k, replace=False)
    center_x = x.take(start_rows)
    center_y = y.take(start_rows)

    clusters = np.zeros(tbl.num_rows)
    for step in np.arange(50):
        # Step A: every point joins the center it is closest to
        new_clusters = make_array()
        for i in np.arange(tbl.num_rows):
            distances = np.sqrt((x.item(i) - center_x)**2 + (y.item(i) - center_y)**2)
            new_clusters = np.append(new_clusters, np.argmin(distances))

        # Step B: every center moves to the average of its points
        for c in np.arange(k):
            center_x[c] = np.mean(x[new_clusters == c])
            center_y[c] = np.mean(y[new_clusters == c])

        # Stop when no point changed clusters
        if step > 0 and np.all(new_clusters == clusters):
            break
        clusters = new_clusters

    return tbl.with_column('cluster', clusters.astype(int))


def plot_clusters(tbl, title):
    """Scatter plot with each cluster in its own color."""
    tbl.scatter(0, 1, group='cluster')
    plt.title(title)

## Part 1: Old Faithful

The same 272 eruptions you met in Lab 8. **Answer handout Q1 before running anything below.**

In [ ]:
faithful = Table.read_table('data/faithful.csv')
faithful.scatter('eruptions', 'waiting')

Now ask k-means for 2 clusters. The `seed` picks which random rows become the starting centers.

In [ ]:
faithful_2 = kmeans(faithful, 2, seed=0)
plot_clusters(faithful_2, 'Old Faithful, k = 2, seed = 0')
faithful_2.group('cluster')

## Part 2: Points with no groups

These 272 points were generated at random, spread evenly over the same ranges as Old Faithful. Nobody built any groups into them.

**Answer handout Q3 before running the k-means cell.**

In [ ]:
noise = Table.read_table('data/noise.csv')
noise.scatter('eruptions', 'waiting')

Adapt the Old Faithful cell above: cluster `noise` with k = 2 and seed = 0, plot it, and show the group sizes.

In [ ]:
noise_2 = ...
plot_clusters(noise_2, 'Random points, k = 2, seed = 0')
noise_2.group('cluster')

### Change the starting point

Run k-means again on **both** datasets with `seed=2`. Record the group sizes and the direction of the boundary in the Q4 table.

In [ ]:
faithful_2b = kmeans(faithful, 2, seed=2)
plot_clusters(faithful_2b, 'Old Faithful, k = 2, seed = 2')
faithful_2b.group('cluster')

In [ ]:
noise_2b = ...
plot_clusters(noise_2b, 'Random points, k = 2, seed = 2')
noise_2b.group('cluster')

## Part 3: Ask for three

Cluster Old Faithful with k = 3, first with `seed=0`, then with `seed=6`. Record the group sizes for handout Q5.

In [ ]:
faithful_3 = kmeans(faithful, ..., seed=0)
plot_clusters(faithful_3, 'Old Faithful, k = 3, seed = 0')
faithful_3.group('cluster')

In [ ]:
faithful_3b = ...
plot_clusters(faithful_3b, 'Old Faithful, k = 3, seed = 6')
faithful_3b.group('cluster')